# 가중치 초기화함수(26. 9. 15)

In [1]:
import torch
print(torch.__version__)              # 2.7.1+cu128
print(torch.cuda.is_available())      # True 나와야 GPU
print(torch.cuda.get_device_name(0))  # RTX 5060

2.7.1+cu128
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [5]:
import torch.nn as nn
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(1, 2),
            nn.Sigmoid()
        )
        self.fc = nn.Linear(2, 1)
        self._init_weights()
    
    def _init_weights(self):
        nn.init.xavier_uniform_(self.layer[0].weight)
        self.layer[0].bias.data.fill_(0.01)

        nn.init.xavier_uniform_(self.fc.weight)
        self.fc.bias.data.fill_(0.01)   # fill → fill_

model = Net().to(device)

In [6]:
for name, param in model.named_parameters():
    print(name, param.data)

layer.0.weight tensor([[ 0.9868],
        [-0.4348]], device='cuda:0')
layer.0.bias tensor([0.0100, 0.0100], device='cuda:0')
fc.weight tensor([[-0.1923, -0.5664]], device='cuda:0')
fc.bias tensor([0.0100], device='cuda:0')


In [9]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(1, 2),
            nn.Sigmoid()
        )
        self.fc = nn.Linear(2, 1)
        self.apply(self._init_weights)
    
    def _init_weights(self, module):          # self, module (인자)
        if isinstance(module, nn.Linear):     # module, nn.Linear
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0.01)

model = Net().to(device)

# L1, L2 규제
-L2를 더 많이 씀

In [10]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
x_data = torch.rand(256, 1) * 10
y_data = 2 * x_data + 1 + torch.randn(256, 1) * 0.5

train_dataset = TensorDataset(x_data, y_data)
train_dataloader = DataLoader(train_dataset, batch_size = 32, shuffle = True, drop_last = True)

model = nn.Linear(1, 1).to(device)
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)


In [13]:
for epoch in range(10):
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        _lambda = 0.5
        # l1_loss = sum(p.abs().sum() for p in model.parameters())
        l2_loss = sum(p.pow(2.0).sum() for p in model.parameters())

        # loss = criterion(output, y) + _lambda * l1_loss
        loss = criterion(output, y) + _lambda * l2_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # print(epoch+1, loss.item(), l1_loss.item())
    print(epoch+1, loss.item(), l2_loss.item())

1 2.9241127967834473 4.895062446594238
2 2.9767820835113525 4.866850852966309
3 3.2405450344085693 4.837184429168701
4 3.278742790222168 4.819674491882324
5 3.0985846519470215 4.802468776702881
6 3.4669711589813232 4.784677982330322
7 3.2158541679382324 4.768472194671631
8 2.802398443222046 4.758174419403076
9 3.5105762481689453 4.7433762550354
10 2.9833321571350098 4.735403060913086


In [14]:
print(model.weight.item(), model.bias.item())

2.1590747833251953 -0.25794732570648193


In [ ]:
# model = nn.Linear(1, 1).to(device)
# optimizer = torch.optim.SGD(model.parameters(), lr = 0.01, weight_decay = 0.01)

In [15]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
x_data = torch.rand(256, 1) * 10
y_data = 2 * x_data + 1 + torch.randn(256, 1) * 0.5

train_dataset = TensorDataset(x_data, y_data)
train_dataloader = DataLoader(train_dataset, batch_size = 32, shuffle = True, drop_last = True)

model = nn.Linear(1, 1).to(device)
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)

In [16]:
for epoch in range(20):
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()

In [17]:
import nltk

In [18]:
for resource in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger','averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet = True)

# 텍스트 증강

In [19]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done.',
    'If a machine is expected to be infalliable, it cannot also be intellingent'
]

aug = naw.ContextualWordEmbsAug(model_path = 'bert-base-uncased', action = 'insert', device = 'cpu')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

c:\Users\wm032\Documents\computervision\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Those who can imagine anything, can create the impossible.
because those who can imagine absolutely anything, can never create for the impossible.
We can only see a short distance ahead, but we can see plenty there that need to be done.
we certainly can only see a safe short distance ahead, but we probably can see plenty in there too that need to first be properly done.
If a machine is expected to be infalliable, it cannot also be intellingent
but if a machine file is still expected to not be infalliable, it either cannot also properly be intellingent


# 랜덤케릭터 (무언가를 지워서 함)

In [21]:
import nlpaug.augmenter.char as nac

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done.',
    'If a machine is expected to be infalliable, it cannot also be intellingent'
]

aug = nac.RandomCharAug(action = 'delete')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Ths who can iagi anything, can cete the mposibe.
We can only see a short distance ahead, but we can see plenty there that need to be done.
We can ol see a short distance ahd, but we can see pley the ta ne to be oe.
If a machine is expected to be infalliable, it cannot also be intellingent
If a aine is expte to be faliabl, it anno as be intellingent


In [22]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done.',
    'If a machine is expected to be infalliable, it cannot also be intellingent'
]

aug = naw.RandomWordAug(action = 'swap')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Who those can anything imagine, can create impossible. the
We can only see a short distance ahead, but we can see plenty there that need to be done.
We can see only a short ahead distance, but can we see there that plenty to need done be.
If a machine is expected to be infalliable, it cannot also be intellingent
A if expected machine is to be infalliable, it also cannot intellingent be


In [23]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done.',
    'If a machine is expected to be infalliable, it cannot also be intellingent'
]

aug = naw.SynonymAug(aug_src = 'wordnet')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those world health organization can reckon anything, can buoy create the impossible.
We can only see a short distance ahead, but we can see plenty there that need to be done.
We send away just visualize a short distance ahead, but we can run into plenty there that need to be do.
If a machine is expected to be infalliable, it cannot also be intellingent
If a machine be expect to be infalliable, information technology cannot also be intellingent


In [25]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done.',
    'If a machine is expected to be infalliable, it cannot also be intellingent'
]

reserved_tokens = [
    ["can", "can't", "cannot", "could"]
]
reserved_aug = naw.ReservedAug(reserved_tokens = reserved_tokens)
# aug = naw.SynonymAug(aug_src = 'wordnet')
augmented_texts = reserved_aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those who can't imagine anything, could create the impossible.
We can only see a short distance ahead, but we can see plenty there that need to be done.
We cannot only see a short distance ahead, but we could see plenty there that need to be done.
If a machine is expected to be infalliable, it cannot also be intellingent
If a machine is expected to be infalliable, it could also be intellingent


In [26]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that need to be done.',
    'If a machine is expected to be infalliable, it cannot also be intellingent'
]

back_translation = naw.BackTranslationAug(
    from_model_name = 'facebook/wmt19-en-de',
    to_model_name = 'facebook/wmt19-de-en',
    device = 'cpu'
)

augmented_texts = back_translation.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

c:\Users\wm032\Documents\computervision\.venv\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wm032\.cache\huggingface\hub\models--facebook--wmt19-en-de. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

vocab-src.json: 0.00B [00:00, ?B/s]

vocab-tgt.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Those who can imagine anything, can create the impossible.
Anyone who can imagine anything can achieve the impossible.
We can only see a short distance ahead, but we can see plenty there that need to be done.
We can only look a little ahead, but we can see a lot there that needs to be done.
If a machine is expected to be infalliable, it cannot also be intellingent
If a machine is expected to be infallible, it cannot be infallible either
